In [108]:
from os import chmod

import pandas as pd
import numpy as np
import networkx as nx
from func.UsefulFunctions import *

In [109]:
#Start with the data for artists

artistDF = pd.read_parquet("Data/Network_artistsWikidata.parquet")


We will be linking artists to their influences, record labels, genres.

In [110]:
artistDF.head()

,artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
0,Q3176190,Q639669,Q6581072,musician,female,Q30,NaN,NaN,NaN,Q43343,https://en.wikipedia.org/wiki/Linda_Gail_Lewis,a2fe484f-4f35-4e0e-a954-2f6d5843b009,Q7544232,Smash Records,United States,folk music,Linda Gail Lewis
1,Q8065552,Q183945,Q6581097,record producer,male,Q145,NaN,NaN,NaN,NaN,https://en.wikipedia.org/wiki/Zalon,19c43212-18cf-4f02-baff-b794de3cdf02,NaN,NaN,United Kingdom,NaN,Zalon
2,Q191819,Q806349,Q6581097,bandleader,male,Q174193,NaN,NaN,NaN,Q9730,https://en.wikipedia.org/wiki/George_Martin,26fa8b67-6c7f-406c-ad64-a1d070092df2,Q21077,Warner Music Group,United Kingdom of Great Britain and Ireland,classical music,George Martin
3,Q21934026,Q183945,Q6581097,record producer,male,NaN,NaN,NaN,NaN,Q37073,https://en.wikipedia.org/wiki/Andrew_Scheps,ee11737c-d50c-4b04-ae5f-56e1f6373116,NaN,NaN,NaN,pop music,Andrew Scheps
4,Q1005957,Q55960555,Q6581097,recording artist,male,Q28,NaN,NaN,NaN,NaN,NaN,6054baf8-3a47-4c3a-8c51-eb976f69ecfc,NaN,NaN,Hungary,NaN,László Földes


In [111]:
# artistDF["Node_Name"] = artistDF["artist"] + "-" +artistDF["artistLabel"]
# artistDF["Genre_Name"] = artistDF["genre"] + "-" +artistDF["genreLabel"]
# artistDF["recordLabel_Name"] = artistDF["recordLabel"] + "-" +artistDF["recordLabelLabel"]
# artistDF["Influence_Name"] = artistDF["influencedBY"] + "-" +artistDF["influencedBYLabel"]


In [112]:
graphInfluence = addToNetwork4(artistDF, "influencedBY", "artist", "has influenced", "Musical Artist", "Musical artist","influencedBYLabel","artistLabel")


In [113]:
graphRecordLabel = addToNetwork4(artistDF, "recordLabel", "artist", "has/had artist","Record Label", "Musical Artist", "recordLabelLabel", "artistLabel")

In [114]:
graphGenre = addToNetwork4(artistDF, "genre", "artist", "has been participated in by", "Genre", "Musical Artist", "genreLabel", "artistLabel")

In [115]:
#compose all the graphs
firstGraph = nx.compose(graphInfluence, graphRecordLabel)
artistGraph = nx.compose(firstGraph, graphGenre)


In [116]:
#See the nodes in artistGraph
artistGraph.nodes["Q122971758"]

{'EntityType': 'Musical Artist', 'Label': 'Ikkimel'}

In [117]:
nx.write_graphml(artistGraph, "./graphs/artistGraph.graphml")

In [118]:
artistGraph.to_undirected().edges("Q5084390")

EdgeDataView([('Q5084390', 'Q42455'), ('Q5084390', 'Q11975'), ('Q5084390', 'Q21077'), ('Q5084390', 'Q202440'), ('Q5084390', 'Q1998195'), ('Q5084390', 'Q2690060'), ('Q5084390', 'Q37073'), ('Q5084390', 'Q187760'), ('Q5084390', 'Q842324'), ('Q5084390', 'Q188450'), ('Q5084390', 'Q487914'), ('Q5084390', 'Q211756'), ('Q5084390', 'Q1298934'), ('Q5084390', 'Q104695865'), ('Q5084390', 'Q9162904')])

In [119]:
ikkimelEgo = nx.ego_graph(artistGraph, "Q122971758", radius=3, center=True, undirected = True).copy()
print(ikkimelEgo.number_of_nodes())

39


In [120]:
sufjanEgo = nx.ego_graph(artistGraph, "Q319502", radius=3, center=True, undirected = True).copy()
print(sufjanEgo.number_of_nodes())
nx.write_graphml(sufjanEgo, "./graphs/sufjanEgo.graphml")

15549


In [121]:
#Subgraph of bjork, fka twigs, electronic music, hip-hop, arca
twigsEgoGraph = nx.ego_graph(artistGraph, "Q15291033", radius=3, center=True, undirected = True).copy()
#electronicEgoGraph = nx.ego_graph(artistGraph, "Q9778", radius=2, center=True,undirected = True).copy()
#subgraph = nx.compose(electronicEgoGraph, bjorkEgoGraph)
print(len(twigsEgoGraph.nodes()))
nx.write_graphml(twigsEgoGraph, "./graphs/twigsEgoGraph.graphml")
nx.write_graphml(ikkimelEgo, "./graphs/ikkimelEgo.graphml")

591


Make a triphop ego network

In [122]:
#Create a triphop ego network for artists of 1 away
trip_hop_WikidataEgo = nx.ego_graph(artistGraph, "Q205560", radius=1, center=True, undirected = True).copy()
nx.write_graphml(trip_hop_WikidataEgo, "./graphs/trip_hop_WikidataEgo.graphml")

In [123]:
#Create a trip hop ego network for from the data from mb
mbPersonData = pl.read_parquet("./Data/Network_personMBData.parquet")
#explode genre list to get only trip hop
mbPersonData = mbPersonData.explode("genreList", empty_as_null=True)
mbPersonData = mbPersonData.unique()
mbPersonData.head()

id,name,type,gender,country,genres,isnis,ipis,life-span,genreList
str,str,str,str,str,list[struct[4]],list[str],list[str],struct[3],str
"""9ce0b294-e25a-4dca-98f0-eaf6d0…","""Hugo Molinares""","""Person""","""Male""","""CO""","[{""7369bb8b-3399-4180-8cb2-4e37e0e7b0e7"",1,"""",""latin""}, {""717f442d-59d1-4349-a14e-c4874acd3509"",1,"""",""salsa""}]",[],[],"{null,null,false}","""salsa"""
"""b1be298a-0b48-43a7-8872-7e402f…","""Bionik""","""Person""","""Male""","""DE""",[],[],[],"{null,null,false}",null
"""21a3af54-0069-4bc8-9d7c-e4c73a…","""Izolda Barudžija""","""Person""","""Female""","""DE""","[{""911c7bbb-172d-4df8-9478-dbff4296e791"",1,"""",""pop""}]",[],[],"{null,""1963"",false}","""pop"""
"""395f7e76-6f4d-437a-be09-70737c…","""Thommy Röben""","""Person""","""Male""","""DE""",[],[],[],"{null,null,false}",null
"""6a96e4bf-280d-4857-9ac1-79562a…","""El Deivy""","""Person""","""Male""","""US""",[],[],[],"{null,""1997-06-21"",false}",null


In [124]:
#On musicbrainz the genre tripHop is called tripHop
tripHopPeople = mbPersonData.filter(
    pl.col("genreList") == "trip hop"
).select(
    pl.col("id"),
    pl.col("name"),
    pl.col("genreList").alias("genreLabel")
).unique()

tripHopPeople.head()

id,name,genreLabel
str,str,str
"""24aeebd1-eb8f-41ad-bc94-dbb087…","""Pogo""","""trip hop"""
"""6a7d6dd8-0f0a-4508-844d-ddea77…","""Interstellar McCartney""","""trip hop"""
"""ef185da6-d061-403b-a75f-517593…","""Lee Curtis Connection""","""trip hop"""
"""234bab50-2f04-4d21-8f91-df93ef…","""Mr. Scruff""","""trip hop"""
"""ae340805-0c01-4172-bff7-cf53db…","""Snowsa""","""trip hop"""


In [125]:
#Lets attempt to find them on wikidata

tripHopPeoplePandas = tripHopPeople.to_pandas()
tripHopPeoplePandas = tripHopPeoplePandas.merge(artistDF[["artist", "musicBrainzID"]], how="left", left_on="id", right_on="musicBrainzID")
tripHopPeoplePandas.drop_duplicates(inplace=True)
tripHopPeoplePandas.head()

,id,name,genreLabel,artist,musicBrainzID
0,24aeebd1-eb8f-41ad-bc94-dbb087868fb6,Pogo,trip hop,Q2830336,24aeebd1-eb8f-41ad-bc94-dbb087868fb6
4,6a7d6dd8-0f0a-4508-844d-ddea779aa9b3,Interstellar McCartney,trip hop,NaN,NaN
5,ef185da6-d061-403b-a75f-517593d9f2bf,Lee Curtis Connection,trip hop,NaN,NaN
6,234bab50-2f04-4d21-8f91-df93ef554855,Mr. Scruff,trip hop,Q933315,234bab50-2f04-4d21-8f91-df93ef554855
11,ae340805-0c01-4172-bff7-cf53db7897d8,Snowsa,trip hop,NaN,NaN


In [126]:
#Make the musicBrainz ego network for tripHOP

#add all the q-number nodes then add all
artistsWithQNumbers= tripHopPeoplePandas[tripHopPeoplePandas.artist.notna()][["artist","name"]].drop_duplicates()
listOfQnumberTuples = artistsWithQNumbers.itertuples(index=False, name=None)
listOfQnumberTuples = [(k, {"Label": v, "EntityType": "Musical Artist-MB"}) for k, v in listOfQnumberTuples]
#then add all the nodes of individuals without q-numbers
artistsWithOnlyID= tripHopPeoplePandas[tripHopPeoplePandas.artist.isna()][["id","name"]].drop_duplicates()
listOfIdNumberTuples = artistsWithOnlyID.itertuples(index=False, name=None)
listOfIdNumberTuples = [(k, {"Label": v, "EntityType": "Musical Artist-MB"}) for k, v in listOfIdNumberTuples]

#connect them all to triphop

mbTripHopEgo = nx.DiGraph()
#add an vertex for triphop
mbTripHopEgo.add_node("Q205560", Label = "trip hop", EntityType = "Genre")
mbTripHopEgo.add_nodes_from(listOfIdNumberTuples)
mbTripHopEgo.add_nodes_from(listOfQnumberTuples)

for x in mbTripHopEgo.nodes:
    #add edges to triphop
    mbTripHopEgo.add_edge("Q205560", x, Type="has been participated in by (MB)")


#save the graph
nx.write_graphml(mbTripHopEgo, "./graphs/mbTripHopEgo.graphml")




In [127]:
mergedTripHop_Graph = nx.compose(mbTripHopEgo, trip_hop_WikidataEgo)
nx.write_graphml(mergedTripHop_Graph, "./graphs/mergedTripHopEgo.graphml")